# Robustness - part 1: break the model on purpose

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tomasvicar/AUI-public/blob/master/labs/robustness/notebooks/ex5_corruption.ipynb)

**Computer-Aided Medical Diagnostics (MPA-PRM)** · lab for the lecture
*Robustness, generalization and reliability*

The ResNet18 of the explainability lab, the same recipe: PneumoniaMNIST
chest X-rays, 28 × 28 pixels, upsampled to 224 × 224, 1500 training images,
three epochs. Today the question is not *where* it looks but **what breaks
it**: four corruptions a real X-ray can suffer, at five severities each.

The training takes a few minutes on a CPU; with *Runtime - Change runtime
type - T4 GPU* it takes seconds. Start it first and read on while it runs.

**Three holes.** The corruption function (two of its four branches), the
loop that measures the accuracy for every corruption and severity, and the
brightness and contrast jitter that is supposed to fix what you find.

# Setup and data

PneumoniaMNIST (Yang et al., *MedMNIST v2*, CC BY 4.0) is downloaded from
Zenodo - 4.2 MB. Labels: 0 normal, 1 pneumonia.

In [ ]:
import hashlib
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
from scipy.ndimage import gaussian_filter

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("running on", DEVICE)

URL = "https://zenodo.org/records/10519652/files/pneumoniamnist.npz?download=1"
MD5 = "28209eda62fecd6e6a2d98b1501bb15f"
SEED = 42
N_TRAIN = 1500
N_EPOCHS = 3
N_EVAL = 200
CLASSES = ["normal", "pneumonia"]


def download(path=Path("pneumoniamnist.npz")):
    """Downloads the 4.2 MB PneumoniaMNIST archive once and checks it."""
    if not path.is_file():
        urllib.request.urlretrieve(URL, path)
    if hashlib.md5(path.read_bytes()).hexdigest() != MD5:
        raise RuntimeError(f"{path} is damaged - delete it and run again")
    return path


def load_pneumonia(path: Path | None = None) -> dict[str, np.ndarray]:
    """Images as float32 arrays (N, 28, 28) in [0, 1], labels as int64 (N,)."""
    archive = np.load(download() if path is None else path)
    data = {}
    for split in ("train", "val", "test"):
        data[f"x_{split}"] = archive[f"{split}_images"].astype(np.float32) / 255.0
        data[f"y_{split}"] = archive[f"{split}_labels"].reshape(-1).astype(np.int64)
    return data


def to_input(images: np.ndarray) -> torch.Tensor:
    """(N, 28, 28) in [0, 1] -> the network input (N, 3, 224, 224), normalized to [-1, 1]."""
    x = torch.from_numpy(np.ascontiguousarray(images, dtype=np.float32)).unsqueeze(1)
    x = F.interpolate(x, size=(224, 224), mode="bilinear", align_corners=False)
    return ((x - 0.5) / 0.5).repeat(1, 3, 1, 1)


data = load_pneumonia()
for split in ("train", "val", "test"):
    print(f"{split:5s} {len(data['y_' + split]):5d} X-rays, "
          f"{data['y_' + split].mean():.0%} pneumonia")

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
for ax, i in zip(axes, range(8)):
    ax.imshow(data["x_test"][i], cmap="gray", vmin=0, vmax=1)
    ax.set_title(CLASSES[data["y_test"][i]], fontsize=9)
    ax.axis("off")
plt.show()

# The model

An ImageNet-pretrained ResNet18 with a new two-class head, fine-tuned whole.
Exactly the code that computed the numbers on the slides.

In [ ]:
def train_resnet(x_train: np.ndarray, y_train: np.ndarray,
                 n_train: int = N_TRAIN, n_epochs: int = N_EPOCHS,
                 device: str = "cpu", verbose: bool = True,
                 seed: int = SEED, augmented: bool = False) -> nn.Module:
    """Fine-tunes an ImageNet ResNet18 on a random subset - the shared recipe.
    The subset is always drawn with SEED; `seed` sets the new layer and the
    order of the batches (the members of a deep ensemble differ only in it);
    `augmented` adds the brightness and contrast jitter."""
    torch.manual_seed(seed)
    subset = np.random.RandomState(SEED).choice(len(x_train), size=n_train, replace=False)
    x, y = x_train[subset], torch.from_numpy(y_train[subset])
    jitter = np.random.RandomState(seed)

    model = tv_models.resnet18(weights=tv_models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, 2)
    model.to(device).train()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    order = torch.Generator().manual_seed(seed)
    for epoch in range(n_epochs):
        total = 0.0
        for batch in torch.randperm(n_train, generator=order).split(32):
            images = x[batch.numpy()]
            if augmented:
                images = augment(images, jitter)
            inputs = to_input(images).to(device)
            loss = F.cross_entropy(model(inputs), y[batch].to(device))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(batch)
        if verbose:
            print(f"epoch {epoch + 1}: loss {total / n_train:.4f}")
    return model.eval()


def predict(model: nn.Module, images: np.ndarray, device: str = "cpu") -> np.ndarray:
    """Probability of pneumonia for every image, in batches of 64."""
    probabilities = []
    with torch.inference_mode():
        for start in range(0, len(images), 64):
            logits = model(to_input(images[start:start + 64]).to(device))
            probabilities.append(logits.softmax(1)[:, 1].cpu().numpy())
    return np.concatenate(probabilities)


model = train_resnet(data["x_train"], data["y_train"], device=DEVICE)
p_test = predict(model, data["x_test"], DEVICE)
print(f"test accuracy, all 624 X-rays: {((p_test > 0.5) == data['y_test']).mean():.3f}")

# Four corruptions

Every corruption is applied to the **28 × 28 image, before the upsampling** -
the noise of a detector is added where the image is taken, not after
somebody enlarged it. Values stay in [0, 1].

| corruption | what it simulates | severity 1 ... 5 |
|---|---|---|
| Gaussian noise | detector noise | standard deviation 0.02 ... 0.12 |
| Gaussian blur | motion, focus | sigma 0.3 ... 1.4 pixels |
| brightness down | underexposure, another scanner | factor 0.9 ... 0.4 |
| contrast down | a flatter image | deviations from the mean × 0.75 ... 0.2 |

### Hole 1 - the two intensity corruptions

In [ ]:
CORRUPTIONS = ["gaussian_noise", "gaussian_blur", "brightness_down", "contrast_down"]
CORRUPTION_LABELS = {"gaussian_noise": "Gaussian noise", "gaussian_blur": "Gaussian blur",
                     "brightness_down": "Brightness down", "contrast_down": "Contrast down"}
SEVERITIES = [0, 1, 2, 3, 4, 5]
NOISE_SIGMA = [0.02, 0.04, 0.06, 0.09, 0.12]
BLUR_SIGMA = [0.3, 0.5, 0.7, 1.0, 1.4]
BRIGHTNESS = [0.9, 0.78, 0.65, 0.52, 0.4]
CONTRAST = [0.75, 0.6, 0.45, 0.32, 0.2]


def corrupt(images, kind, severity, seed=0):
    """One corruption at one severity (0 = clean) on (N, 28, 28) images in [0, 1]."""
    if severity == 0:
        return images
    level = severity - 1
    if kind == "gaussian_noise":
        rng = np.random.default_rng(seed)
        out = images + rng.normal(0.0, NOISE_SIGMA[level], size=images.shape)
    elif kind == "gaussian_blur":
        out = gaussian_filter(images, sigma=(0, BLUR_SIGMA[level], BLUR_SIGMA[level]))
    elif kind == "brightness_down":
        # TODO 1a: multiply the images by the factor BRIGHTNESS[level]
        out = ...
    elif kind == "contrast_down":
        # TODO 1b: pull every image towards its OWN mean intensity - keep the
        #          mean, multiply the deviations from it by CONTRAST[level].
        #          Hint: images.mean(axis=(1, 2), keepdims=True)
        out = ...
    else:
        raise ValueError(kind)
    return np.clip(out, 0.0, 1.0).astype(np.float32)

In [ ]:
# One test X-ray under every corruption and severity.
def evaluation_subset(x_test: np.ndarray, y_test: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """The fixed 200 test images every severity is compared on."""
    index = np.random.RandomState(SEED).choice(len(x_test), size=N_EVAL, replace=False)
    return x_test[index], y_test[index]


images, labels = evaluation_subset(data["x_test"], data["y_test"])
fig, axes = plt.subplots(len(CORRUPTIONS), len(SEVERITIES), figsize=(10, 7))
for row, kind in enumerate(CORRUPTIONS):
    for col, severity in enumerate(SEVERITIES):
        axes[row, col].imshow(corrupt(images[:1], kind, severity)[0], cmap="gray", vmin=0, vmax=1)
        axes[row, col].set_xticks([]); axes[row, col].set_yticks([])
        if row == 0:
            axes[row, col].set_title("clean" if severity == 0 else f"severity {severity}")
        if col == 0:
            axes[row, col].set_ylabel(CORRUPTION_LABELS[kind])
plt.show()

# How much accuracy does each corruption cost?

Before you run the next cell, **write down a bet**: which of the four rows
above will hurt the model most?

### Hole 2 - the accuracy table

In [ ]:
# TODO 2: the accuracy of the model for every corruption at every severity,
#         always on the same 200 X-rays (`images`, `labels` from the cell above).
#
#   results[kind] = [accuracy at severity 0, 1, ..., 5]
#
# `predict(model, some_images, DEVICE)` returns the probability of pneumonia;
# the prediction is `p > 0.5`. Print one row per corruption.

results = {}
...

In [ ]:
majority = labels.mean()
plt.figure(figsize=(7, 4.5))
for kind, accuracies in results.items():
    plt.plot(SEVERITIES, accuracies, "o-", label=CORRUPTION_LABELS[kind])
plt.axhline(majority, color="k", ls="--", lw=1, label=f"always 'pneumonia' = {majority:.2f}")
plt.xlabel("severity (0 = clean)"); plt.ylabel("accuracy on 200 test X-rays")
plt.legend(); plt.show()

# Train for the change you expect

The model never saw a darker or flatter X-ray in training. The cheapest fix
is to show it some: every training image gets a random brightness and a
random contrast, so the variation is seen before deployment.

### Hole 3 - brightness and contrast jitter

In [ ]:
# TODO 3: brightness and contrast jitter for training. Every image of a batch
#         gets its own random brightness factor, drawn uniformly from
#         AUGMENT_BRIGHTNESS, and its own contrast factor from AUGMENT_CONTRAST,
#         applied as in hole 1: contrast around the image's own mean first,
#         then brightness; clip to [0, 1]. `rng` is a np.random.RandomState -
#         draw with rng.uniform(low, high, size=(n, 1, 1)), the brightness
#         factors first, then the contrast factors (the order of the draws
#         decides which numbers you get).
#         `train_resnet(..., augmented=True)` calls it on every batch.

AUGMENT_BRIGHTNESS = (0.4, 1.0)
AUGMENT_CONTRAST = (0.2, 1.0)


def augment(images, rng):
    ...

In [ ]:
# The same recipe once more, now with the jitter - a few minutes on a CPU.
model_augmented = train_resnet(data["x_train"], data["y_train"], device=DEVICE, augmented=True)

results_augmented = {}
for kind in CORRUPTIONS:
    results_augmented[kind] = []
    for severity in SEVERITIES:
        p = predict(model_augmented, corrupt(images, kind, severity), DEVICE)
        results_augmented[kind].append(float(((p > 0.5) == labels).mean()))
    print(f"{kind:16s}", " ".join(f"{a:.3f}" for a in results_augmented[kind]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2), sharey=True)
for ax, table, title in [(axes[0], results, "plain"), (axes[1], results_augmented, "with jitter")]:
    for kind, accuracies in table.items():
        ax.plot(SEVERITIES, accuracies, "o-", label=CORRUPTION_LABELS[kind])
    ax.axhline(labels.mean(), color="k", ls="--", lw=1)
    ax.set(title=title, xlabel="severity (0 = clean)")
axes[0].set_ylabel("accuracy on 200 test X-rays"); axes[1].legend(); plt.show()

# The price on clean X-rays - all 624 of them: accuracy at 0.5, and the ranking
# (AUC - the area under the ROC curve).
from sklearn.metrics import roc_auc_score
p_augmented = predict(model_augmented, data["x_test"], DEVICE)
for name, p in [("plain", p_test), ("with jitter", p_augmented)]:
    print(f"{name:12s} accuracy {((p > 0.5) == data['y_test']).mean():.3f}   "
          f"AUC {roc_auc_score(data['y_test'], p):.3f}   mean P(pneumonia) {p.mean():.3f}")

# Does the model notice?

A model that knows it is out of its depth could flag the X-ray for a human.
The simplest flag is a low confidence. Does it fire on the darker X-rays?

In [ ]:
# Does the plain model notice that the X-rays got darker? Its confidence is the
# probability of the label it predicts. A flag fires below the confidence that
# only 5 % of the VALIDATION X-rays fall under.
def confidence(p):
    return np.maximum(p, 1 - p)


threshold = np.quantile(confidence(predict(model, data["x_val"], DEVICE)), 0.05)
darker = corrupt(data["x_test"], "brightness_down", 3)          # 35 % darker
for name, x in [("clean", data["x_test"]), ("35 % darker", darker)]:
    p = predict(model, x, DEVICE)
    wrong = (p > 0.5) != data["y_test"]
    flagged = confidence(p) < threshold
    print(f"{name:12s} wrong {wrong.mean():.0%}   flagged {flagged.mean():.0%}   "
          f"errors caught {flagged[wrong].mean():.0%}   mean confidence {confidence(p).mean():.3f}")

# Questions

1. Which corruption hurts most, and which least? Did your bet hold?
2. What is the model doing when its accuracy sits exactly on the dashed line?
3. Pick a threshold below which you would call the model clinically useless
   and justify it. At which severity does each corruption cross it?
4. What did the jitter buy, and what did it cost on clean X-rays? Why did the
   AUC stay where it was while the accuracy at 0.5 fell?
5. Noise and blur did not get better with the jitter, and brightness falls
   again at severity 5. Why?
6. On the darker X-rays the model is wrong far more often - and flags fewer of
   them. Why does a low-confidence flag stay silent exactly here?

In [ ]:
# Space for your answers - e.g. compare results and results_augmented row by row.

---

*The numbers are computed by
[`pneumonia.py`](https://github.com/tomasvicar/AUI-public/blob/master/lectures/robustness/code/pneumonia.py) of the lecture. The
notebook is built by [`code/build_notebooks.py`](https://github.com/tomasvicar/AUI/blob/master/labs/robustness/code/build_notebooks.py) -
editing the `.ipynb` by hand gets overwritten.*